# Credit Card Fraud Detection: Exploratory Data Analysis & Problem Framing

## 1. Business Problem Context & Asymmetric Cost Framing

> **Business Problem Formulation**  
> In modern digital payment ecosystems, credit card fraud represents billions of dollars in annual losses while accounting for fewer than 2 out of every 1,000 transactions ($<0.2\%$). Standard machine learning classifiers tuned for raw accuracy degrade catastrophically under this extreme $0.172\%$ class imbalance: a naive null model predicting every transaction as legitimate achieves $>99.828\%$ accuracy while failing to intercept a single dollar of fraudulent activity.
>
> Furthermore, blanket risk-aversion introduces operational friction by declining legitimate cardholders, causing customer attrition and heavy review costs. This system formulates fraud detection as an asymmetric financial risk minimization problem: pairing cost-sensitive gradient boosted ensembles with post-hoc probability calibration to directly optimize the operational trade-off between chargeback recovery and review friction.

### Cost Matrix Definition
- **False Negative (FN):** Missed fraud transaction. $\text{Cost} = \text{Amount}(x) + \$25.00$ (chargeback fee + operational reissuance).
- **False Positive (FP):** Legitimate transaction declined. $\text{Cost} = \$10.00$ (cardholder SMS verification, friction, and potential churn).
- **True Positive (TP):** Flagged and prevented fraud. $\text{Cost} \approx \$0.00$.
- **True Negative (TN):** Frictionless approval. $\text{Cost} = \$0.00$.

$$\mathcal{L}_{\text{business}}(\tau) = \sum_{i=1}^N \left[ y_i \cdot \mathbb{I}(\hat{p}_i < \tau) \cdot (\text{Amount}_i + 25) + (1 - y_i) \cdot \mathbb{I}(\hat{p}_i \ge \tau) \cdot 10 \right]$$

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.config import get_settings
from src.data.load import load_dataset
from src.eda.run_eda import run_full_eda

settings = get_settings()
print(f"Project: {settings.project.name} v{settings.project.version}")

## 2. Dataset Ingestion and Schema Audit

In [ ]:
df = load_dataset(settings)
print(f"Dataset shape: {df.shape}")
df.head()

## 3. Class Imbalance Quantification

In [ ]:
counts = df['Class'].value_counts()
print(f"Legitimate (0): {counts[0]:,} ({counts[0]/len(df)*100:.3f}%)")
print(f"Fraudulent (1): {counts[1]:,} ({counts[1]/len(df)*100:.3f}%)")
print(f"Imbalance Ratio: 1:{int(counts[0]/counts[1])}")

## 4. Transaction Amount & Skewness Analysis

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
sns.kdeplot(np.log1p(df[df['Class'] == 0]['Amount']), label='Legitimate', ax=ax[0], fill=True)
sns.kdeplot(np.log1p(df[df['Class'] == 1]['Amount']), label='Fraud', ax=ax[0], fill=True, color='red')
ax[0].set_title('Log-Transformed Amount Distribution')
ax[0].set_xlabel('log1p(Amount)')
ax[0].legend()

sns.boxplot(data=df, x='Class', y='Amount', showfliers=False, ax=ax[1], palette=['#2b5c8f', '#d9534f'])
ax[1].set_title('Transaction Amount Boxplot (No Fliers)')
plt.show()

## 5. Temporal Patterns & Diurnal Cyclicality

In [ ]:
hours = ((df['Time'] % 86400) / 3600).astype(int)
hourly = pd.DataFrame({'Hour': hours, 'Class': df['Class']})
hourly_rate = hourly.groupby('Hour')['Class'].mean() * 100

plt.figure(figsize=(10, 4))
plt.plot(hourly_rate.index, hourly_rate.values, marker='o', color='#d9534f', lw=2)
plt.title('Fraud Probability by Hour of Day (24-Hour Cycle)')
plt.xlabel('Hour of Day')
plt.ylabel('Fraud Rate (%)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 6. Spearman Rank Correlation Analysis with Fraud

In [ ]:
v_cols = [c for c in df.columns if c.startswith('V')]
corrs = df[v_cols].apply(lambda s: s.corr(df['Class'], method='spearman'))
corrs_sorted = corrs.sort_values()
top_corrs = pd.concat([corrs_sorted.head(5), corrs_sorted.tail(5)])

plt.figure(figsize=(8, 5))
colors = ['#2b5c8f' if v < 0 else '#d9534f' for v in top_corrs.values]
plt.barh(top_corrs.index, top_corrs.values, color=colors)
plt.title('Top 10 Features Correlated with Fraud (Spearman Rank)')
plt.xlabel('Correlation')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()

## 7. Run Full Automated Audit & Generate Figures

In [ ]:
summary = run_full_eda(df=df, config=settings)
print("EDA Audit summary:", summary)